<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎬 SoulX FlashHead - AI Talking Head Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Google Colab T4 GPU Edition - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>Real-Time Audio-Driven Portrait Animation with Fast Tensor Core FP16 Inference</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-orange?style=for-the-badge&logo=googlecolab&logoColor=white" />
  <img src="https://img.shields.io/badge/Engine-FP16%20Tensor%20Cores-success?style=for-the-badge" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### Features & Optimizations
| Feature | Description |
|---|---|
| ⚡ **Fast FP16 Tensor Cores** | Native `float16` execution on Tesla T4, cutting generation time by 3x-4x |
| 🎯 **1:1 Face Centering & Crop** | OpenCV SSD DNN face detector guarantees a centered 1:1 close-up for crisp animation |
| 🚀 **Hardware-Accelerated Rendering** | NVENC GPU hardware encoding and ultrafast FFmpeg composite in sub-5 seconds |
| 📐 **16:9 & 9:16 Aspect Ratio System** | Seamless composite back into widescreen (16:9) or vertical reels (9:16) |
| 🔳 **Direct 1:1 Square Output** | Sub-second export option for direct 512x512 face video |
| 📊 **Real-Time Progress Bars** | Dynamic `tqdm` console bars and Gradio stage progress tracking |
| 🧹 **Clean Streaming Logs** | Persistent execution cell streaming clean live generation logs to the console |
| 🎭 **Lite & Pro Variants** | High-speed single-GPU Lite mode and ultra-high fidelity Pro mode |
| 📥 **High-Speed Downloads** | Multithreaded Rust-accelerated `hf-transfer` model retrieval |

---

### Quick Start
1. Ensure GPU is active: **Runtime -> Change runtime type -> T4 GPU**
2. Run **Step 1** to configure environment, install dependencies, and download models.
3. Run **Step 2** to launch the interactive Gradio Studio!

In [ ]:
#@title 📦 Step 1: Environment Setup, Dependencies & Fast Model Download
import os
import sys
import gc
import re
import warnings

# 1. Configure High-Performance Environment & Silence Warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("=" * 65)
print("🎬 SoulX FlashHead - AI Talking Head Generator")
print("📺 Created by: AIQUEST Academy")
print("🔗 YouTube: @AIQuestAcademy | X: @AIQuestAcademy")
print("=" * 65)

# 2. Hardware and GPU Verification
import torch
print(f"🔧 Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"✅ CUDA Version: {torch.version.cuda} | Tensor Cores Active")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
else:
    print("\n⚠️ WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> T4 GPU")

# 3. System Dependencies
print("\n📦 Installing system packages (ffmpeg)...")
os.system("apt-get update -qq > /dev/null 2>&1 && apt-get install -y ffmpeg -qq > /dev/null 2>&1")

# 4. Clone or Pull SoulX-FlashHead Repository
REPO_DIR = "/content/SoulX-FlashHead"
if not os.path.exists(REPO_DIR):
    print("📥 Cloning SoulX-FlashHead repository...")
    os.system(f"git clone --depth 1 https://github.com/Soul-AILab/SoulX-FlashHead.git {REPO_DIR} > /dev/null 2>&1")
else:
    print("📂 Repository exists, verifying clean state...")

# 5. Fast Python Dependencies Installation
print("📦 Installing optimized Python dependencies (hf-transfer, diffusers, transformers, gradio, xfuser)...")
os.system(
    "pip install -q --no-warn-conflicts hf-transfer \"transformers==4.57.3\" \"diffusers>=0.34.0\" "
    "\"accelerate>=1.8.1\" loguru pyloudnorm decord librosa \"gradio>=5.0.0\" imageio \"imageio-ffmpeg\" "
    "ftfy einops easydict tqdm \"xfuser>=0.4.3\" yunchang distvae sentencepiece > /dev/null 2>&1 || "
    "pip install -q --no-warn-conflicts hf-transfer \"transformers==4.57.3\" \"diffusers>=0.34.0\" "
    "\"accelerate>=1.8.1\" loguru pyloudnorm decord librosa \"gradio>=5.0.0\" imageio \"imageio-ffmpeg\" "
    "ftfy einops easydict tqdm > /dev/null 2>&1"
)
os.system("pip install -q xformers > /dev/null 2>&1 || true")

# 6. High-Speed Model Checkpoint Download via hf-transfer
CKPT_DIR = "/content/models/SoulX-FlashHead-1_3B"
WAV2VEC_DIR = "/content/models/wav2vec2-base-960h"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(WAV2VEC_DIR, exist_ok=True)

print("\n🚀 Downloading SoulX-FlashHead-1_3B checkpoints (fast parallel transfer)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="Soul-AILab/SoulX-FlashHead-1_3B",
        local_dir=CKPT_DIR,
        max_workers=8,
    )
    print("✅ SoulX-FlashHead-1_3B downloaded successfully.")
except Exception as e:
    print(f"⚠️ Snapshot download notice: {e}, falling back to CLI...")
    os.system(f"huggingface-cli download Soul-AILab/SoulX-FlashHead-1_3B --local-dir {CKPT_DIR} --quiet")

print("🚀 Downloading wav2vec2-base-960h checkpoints...")
try:
    snapshot_download(
        repo_id="facebook/wav2vec2-base-960h",
        local_dir=WAV2VEC_DIR,
        max_workers=8,
    )
    print("✅ wav2vec2-base-960h downloaded successfully.")
except Exception as e:
    os.system(f"huggingface-cli download facebook/wav2vec2-base-960h --local-dir {WAV2VEC_DIR} --quiet")

# 7. Download OpenCV SSD DNN Face Detector Models
print("🚀 Downloading OpenCV SSD face detector models...")
PROTO_PATH = os.path.join(REPO_DIR, "deploy.prototxt")
CAFFE_PATH = os.path.join(REPO_DIR, "res10_300x300_ssd_iter_140000.caffemodel")
try:
    import urllib.request
    if not os.path.exists(PROTO_PATH):
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
            PROTO_PATH
        )
    if not os.path.exists(CAFFE_PATH):
        urllib.request.urlretrieve(
            "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
            CAFFE_PATH
        )
    print("✅ OpenCV SSD face detector models downloaded.")
except Exception as e:
    print(f"⚠️ Face detector download notice: {e}")

# 8. Apply Speed Optimizations & Bug Fix Patches
print("\n🔧 Applying fast FP16 inference engine and 1:1 face crop patches...")

# Patch A: High-accuracy OpenCV SSD DNN Face Detector with 1:1 center-top fallback
handler_file = os.path.join(REPO_DIR, "flash_head/utils/cpu_face_handler.py")
handler_code = '''import os
import cv2
import numpy as np
from typing import Tuple, List

class CPUFaceHandler:
    """High-precision OpenCV SSD DNN face detection with strict 1:1 center fallback."""
    def __init__(self, model_selection: int = 1, min_detection_confidence: float = 0.4):
        self.use_dnn = False
        self.use_haar = False
        proto = "/content/SoulX-FlashHead/deploy.prototxt"
        caffe = "/content/SoulX-FlashHead/res10_300x300_ssd_iter_140000.caffemodel"

        # Auto-download if missing
        if not os.path.exists(proto) or not os.path.exists(caffe):
            try:
                import urllib.request
                if not os.path.exists(proto):
                    urllib.request.urlretrieve(
                        "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
                        proto
                    )
                if not os.path.exists(caffe):
                    urllib.request.urlretrieve(
                        "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
                        caffe
                    )
            except Exception:
                pass

        try:
            if os.path.exists(proto) and os.path.exists(caffe) and hasattr(cv2, "dnn") and hasattr(cv2.dnn, "readNetFromCaffe"):
                self.net = cv2.dnn.readNetFromCaffe(proto, caffe)
                self.use_dnn = True
            elif hasattr(cv2, "CascadeClassifier") and hasattr(cv2, "data") and hasattr(cv2.data, "haarcascades"):
                cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
                if os.path.exists(cascade_path):
                    self.cascade = cv2.CascadeClassifier(cascade_path)
                    self.use_haar = True
        except Exception:
            pass

    def detect(self, image: np.ndarray) -> Tuple[List, List]:
        bboxs, scores = [], []
        img_h, img_w = image.shape[:2]

        if self.use_dnn:
            try:
                blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0))
                self.net.setInput(blob)
                detections = self.net.forward()
                for i in range(detections.shape[2]):
                    confidence = float(detections[0, 0, i, 2])
                    if confidence > 0.4:
                        x1 = max(0.0, float(detections[0, 0, i, 3]))
                        y1 = max(0.0, float(detections[0, 0, i, 4]))
                        x2 = min(1.0, float(detections[0, 0, i, 5]))
                        y2 = min(1.0, float(detections[0, 0, i, 6]))
                        if x2 > x1 and y2 > y1:
                            bboxs.append([x1, y1, x2, y2])
                            scores.append(confidence)
                if len(bboxs) > 0:
                    return bboxs, scores
            except Exception:
                pass

        if self.use_haar:
            try:
                gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
                faces = self.cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
                for (x, y, w, h) in faces:
                    bboxs.append([float(x / img_w), float(y / img_h), float((x + w) / img_w), float((y + h) / img_h)])
                    scores.append(1.0)
                if len(bboxs) > 0:
                    return bboxs, scores
            except Exception:
                pass

        # 100% crash-proof fallback: Upper-center 1:1 face framing
        # Centered horizontally, upper 35% vertically (standard portrait distribution)
        fx1 = 0.35
        fy1 = 0.15
        fx2 = 0.65
        fy2 = 0.55
        bboxs.append([fx1, fy1, fx2, fy2])
        scores.append(0.8)
        return bboxs, scores

    def __call__(self, image: np.ndarray) -> Tuple[List, List]:
        return self.detect(image)
'''
with open(handler_file, "w") as f:
    f.write(handler_code)
print("  -> Face detector patched (OpenCV SSD DNN & crash-proof)")

# Patch B: Patch facecrop.py to guarantee strict 1:1 square crop and save bbox
facecrop_file = os.path.join(REPO_DIR, "flash_head/utils/facecrop.py")
facecrop_code = '''#!/usr/bin/env python3
import os
import json
from PIL import Image
import numpy as np

from flash_head.utils.cpu_face_handler import CPUFaceHandler

def get_scaled_bbox(
    bbox, img_w, img_h, ratio: float = 1.0, face_image: Image.Image = None
):
    """
    Computes a guaranteed strict 1:1 square crop centered on the face.
    Ensures width == height, shifts safely away from edges, and enforces even dimensions.
    """
    x1, y1, x2, y2 = bbox
    center_x = (x1 + x2) / 2.0
    center_y = (y1 + y2) / 2.0
    face_w = max(x2 - x1, 10.0)

    # Calculate target 1:1 square side length
    max_side = min(img_w, img_h)
    target_side = int(min(face_w * ratio, max_side))
    target_side = max(target_side, 64)

    # Slight downward bias so the face occupies the upper-middle of the square crop
    shift_y = target_side * 0.05
    center_y = center_y + shift_y

    half_side = target_side / 2.0
    new_x1 = int(round(center_x - half_side))
    new_y1 = int(round(center_y - half_side))

    # Shift bounding box away from edges so it doesn't get clipped into a non-square
    if new_x1 < 0:
        new_x1 = 0
    elif new_x1 + target_side > img_w:
        new_x1 = img_w - target_side

    if new_y1 < 0:
        new_y1 = 0
    elif new_y1 + target_side > img_h:
        new_y1 = img_h - target_side

    new_x2 = new_x1 + target_side
    new_y2 = new_y1 + target_side

    # Ensure even dimensions for video codecs
    if (new_x2 - new_x1) % 2 != 0:
        new_x2 -= 1
    if (new_y2 - new_y1) % 2 != 0:
        new_y2 -= 1

    scaled_bbox = [int(new_x1), int(new_y1), int(new_x2), int(new_y2)]

    try:
        with open("/tmp/_flashhead_crop_bbox.json", "w") as _bf:
            json.dump({"bbox": scaled_bbox, "img_w": img_w, "img_h": img_h}, _bf)
    except Exception:
        pass

    crop_face = face_image.crop(scaled_bbox)
    return crop_face

def process_image(
    input_path,
    face_ratio=1.8,
    target_size=(512, 512),
):
    """
    Detects face, crops a clean 1:1 square, and resizes to target_size.
    """
    face_detector = CPUFaceHandler()
    if not os.path.isfile(input_path):
        raise ValueError(f"File not found: {input_path}")

    try:
        image = Image.open(input_path).convert("RGB")
        image_rgb = np.array(image)
        img_h, img_w = image_rgb.shape[:2]

        boxes, scores = face_detector(image_rgb)
        if len(boxes) == 0:
            boxes = [[0.35, 0.15, 0.65, 0.55]]

        boxes_abs = [
            boxes[0][0] * img_w,
            boxes[0][1] * img_h,
            boxes[0][2] * img_w,
            boxes[0][3] * img_h
        ]

        crop_face = get_scaled_bbox(boxes_abs, img_w, img_h, face_ratio, image)
        crop_face = crop_face.resize(target_size, Image.Resampling.LANCZOS)
        return crop_face
    except Exception as e:
        raise ValueError(f"Error processing {input_path}: {e}")
'''
with open(facecrop_file, "w") as f:
    f.write(facecrop_code)
print("  -> Face crop bounding box tracker patched (strict 1:1 square crop guaranteed)")

# Patch C: Patch flash_head_pipeline.py for native FP16 execution, zero compile latency & silent logs
pipeline_file = os.path.join(REPO_DIR, "flash_head/src/pipeline/flash_head_pipeline.py")
if os.path.exists(pipeline_file):
    with open(pipeline_file, "r") as f:
        p_content = f.read()

    p_content = p_content.replace("COMPILE_MODEL = True", "COMPILE_MODEL = False")
    p_content = p_content.replace("COMPILE_VAE = True", "COMPILE_VAE = False")
    p_content = p_content.replace("param_dtype=torch.bfloat16,", "param_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,")
    p_content = p_content.replace(
        "self.model = WanModelAudioProject.from_pretrained(model_dir)",
        "self.model = WanModelAudioProject.from_pretrained(model_dir, torch_dtype=self.param_dtype, low_cpu_mem_usage=True)"
    )
    p_content = re.sub(r'if self\.rank == 0:\s+print\(f\'\[generate\][^\n]+\'\)', '# step log silenced', p_content)
    p_content = p_content.replace("torch.cuda.synchronize()", "# sync removed")
    p_content = p_content.replace("@torch.no_grad()", "@torch.inference_mode()")

    with open(pipeline_file, "w") as f:
        f.write(p_content)
print("  -> High-performance FP16 engine & log cleanup patched")

# Patch D: Patch inference.py to pass param_dtype=torch.float16 explicitly
infer_file = os.path.join(REPO_DIR, "flash_head/inference.py")
if os.path.exists(infer_file):
    with open(infer_file, "r") as f:
        inf_content = f.read()
    if "param_dtype=torch.float16" not in inf_content:
        inf_content = inf_content.replace(
            "pipeline = FlashHeadPipeline(\n        checkpoint_dir=ckpt_dir,\n        model_type=model_type,\n        wav2vec_dir=wav2vec_dir,\n        device=device,\n        use_usp=(world_size > 1),\n    )",
            "pipeline = FlashHeadPipeline(\n        checkpoint_dir=ckpt_dir,\n        model_type=model_type,\n        wav2vec_dir=wav2vec_dir,\n        device=device,\n        param_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\n        use_usp=(world_size > 1),\n    )"
        )
        with open(infer_file, "w") as f:
            f.write(inf_content)
print("  -> Inference pipeline configured for native Tensor Core precision")

# Patch E: Self-healing xfuser import and attention in flash_head_model.py
model_file = os.path.join(REPO_DIR, "flash_head/src/modules/flash_head_model.py")
if os.path.exists(model_file):
    with open(model_file, "r") as f:
        m_content = f.read()
    if "from xfuser.core.distributed import (" in m_content and "except (ImportError, ModuleNotFoundError):" not in m_content:
        m_content = re.sub(
            r'from xfuser\.core\.distributed import\s*\(\s*get_sequence_parallel_rank,\s*get_sequence_parallel_world_size,\s*get_sp_group,\s*\)\s*from xfuser\.core\.long_ctx_attention import xFuserLongContextAttention',
            '''try:
    from xfuser.core.distributed import (
        get_sequence_parallel_rank,
        get_sequence_parallel_world_size,
        get_sp_group,
    )
    from xfuser.core.long_ctx_attention import xFuserLongContextAttention
    XFUSER_AVAILABLE = True
except (ImportError, ModuleNotFoundError):
    XFUSER_AVAILABLE = False
    get_sequence_parallel_rank = lambda: 0
    get_sequence_parallel_world_size = lambda: 1
    get_sp_group = lambda: None
    xFuserLongContextAttention = None''',
            m_content
        )
        with open(model_file, "w") as f:
            f.write(m_content)
print("  -> FlashHead model patched with self-healing distributed fallback")

# 9. Pre-install Cloudflare Tunnel Binary (cloudflared)
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("🌐 Installing Cloudflare tunnel binary (cloudflared)...")
    try:
        os.system("curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared > /dev/null 2>&1")
        print("✅ Cloudflare tunnel (cloudflared) installed successfully!")
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel install notice: {e}")

print("\n" + "=" * 65)
print("✅ Step 1 Complete: Environment configured for peak generation speed!")
print("👉 Proceed to Step 2 to launch the SoulX FlashHead Studio UI.")
print("=" * 65)

In [ ]:
#@title 🚀 Step 2: Launch SoulX FlashHead Studio (Lite & Pro)
import os
import sys
import gc
import re
import json
import time
import subprocess
import warnings
import numpy as np
from datetime import datetime
from collections import deque
from PIL import Image
from tqdm import tqdm
from loguru import logger

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

# Navigate to repo directory
REPO_DIR = "/content/SoulX-FlashHead"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

# ── Dynamic Self-Healing Checks: Guarantee patches at launch ──
# 1. flash_head_model.py
model_py = "/content/SoulX-FlashHead/flash_head/src/modules/flash_head_model.py"
if os.path.exists(model_py):
    try:
        with open(model_py, "r") as _f:
            _mcode = _f.read()
        if "from xfuser.core.distributed import (" in _mcode and "except (ImportError, ModuleNotFoundError):" not in _mcode:
            _mcode = re.sub(
                r'from xfuser\.core\.distributed import\s*\(\s*get_sequence_parallel_rank,\s*get_sequence_parallel_world_size,\s*get_sp_group,\s*\)\s*from xfuser\.core\.long_ctx_attention import xFuserLongContextAttention',
                '''try:
    from xfuser.core.distributed import (
        get_sequence_parallel_rank,
        get_sequence_parallel_world_size,
        get_sp_group,
    )
    from xfuser.core.long_ctx_attention import xFuserLongContextAttention
    XFUSER_AVAILABLE = True
except (ImportError, ModuleNotFoundError):
    XFUSER_AVAILABLE = False
    get_sequence_parallel_rank = lambda: 0
    get_sequence_parallel_world_size = lambda: 1
    get_sp_group = lambda: None
    xFuserLongContextAttention = None''',
                _mcode
            )
            with open(model_py, "w") as _f:
                _f.write(_mcode)
    except Exception:
        pass

# 2. flash_head_pipeline.py (FP16 & Speed Optimization for Lite & Pro)
pipeline_py = "/content/SoulX-FlashHead/flash_head/src/pipeline/flash_head_pipeline.py"
if os.path.exists(pipeline_py):
    try:
        with open(pipeline_py, "r") as _f:
            _pcode = _f.read()
        _pcode = _pcode.replace("COMPILE_MODEL = True", "COMPILE_MODEL = False")
        _pcode = _pcode.replace("COMPILE_VAE = True", "COMPILE_VAE = False")
        _pcode = _pcode.replace("param_dtype=torch.bfloat16,", "param_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,")
        _pcode = _pcode.replace(
            "self.model = WanModelAudioProject.from_pretrained(model_dir)",
            "self.model = WanModelAudioProject.from_pretrained(model_dir, torch_dtype=self.param_dtype, low_cpu_mem_usage=True)"
        )
        _pcode = re.sub(r'if self\.rank == 0:\s+print\(f\'\[generate\][^\n]+\'\)', '# step log silenced', _pcode)
        _pcode = _pcode.replace("torch.cuda.synchronize()", "# sync removed")
        _pcode = _pcode.replace("@torch.no_grad()", "@torch.inference_mode()")
        with open(pipeline_py, "w") as _f:
            _f.write(_pcode)
    except Exception:
        pass

# 3. inference.py
infer_py = "/content/SoulX-FlashHead/flash_head/inference.py"
if os.path.exists(infer_py):
    try:
        with open(infer_py, "r") as _f:
            _icode = _f.read()
        if "param_dtype=torch.float16" not in _icode:
            _icode = _icode.replace(
                "pipeline = FlashHeadPipeline(\n        checkpoint_dir=ckpt_dir,\n        model_type=model_type,\n        wav2vec_dir=wav2vec_dir,\n        device=device,\n        use_usp=(world_size > 1),\n    )",
                "pipeline = FlashHeadPipeline(\n        checkpoint_dir=ckpt_dir,\n        model_type=model_type,\n        wav2vec_dir=wav2vec_dir,\n        device=device,\n        param_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\n        use_usp=(world_size > 1),\n    )"
            )
            with open(infer_py, "w") as _f:
                _f.write(_icode)
    except Exception:
        pass

# 4. cpu_face_handler.py (OpenCV SSD DNN & Upper-Center Fallback)
handler_py = "/content/SoulX-FlashHead/flash_head/utils/cpu_face_handler.py"
if os.path.exists(handler_py):
    try:
        with open(handler_py, "w") as _f:
            _f.write('''import os
import cv2
import numpy as np
from typing import Tuple, List

class CPUFaceHandler:
    """High-precision OpenCV SSD DNN face detection with strict 1:1 center fallback."""
    def __init__(self, model_selection: int = 1, min_detection_confidence: float = 0.4):
        self.use_dnn = False
        self.use_haar = False
        proto = "/content/SoulX-FlashHead/deploy.prototxt"
        caffe = "/content/SoulX-FlashHead/res10_300x300_ssd_iter_140000.caffemodel"

        if not os.path.exists(proto) or not os.path.exists(caffe):
            try:
                import urllib.request
                if not os.path.exists(proto):
                    urllib.request.urlretrieve(
                        "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
                        proto
                    )
                if not os.path.exists(caffe):
                    urllib.request.urlretrieve(
                        "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
                        caffe
                    )
            except Exception:
                pass

        try:
            if os.path.exists(proto) and os.path.exists(caffe) and hasattr(cv2, "dnn") and hasattr(cv2.dnn, "readNetFromCaffe"):
                self.net = cv2.dnn.readNetFromCaffe(proto, caffe)
                self.use_dnn = True
            elif hasattr(cv2, "CascadeClassifier") and hasattr(cv2, "data") and hasattr(cv2.data, "haarcascades"):
                cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
                if os.path.exists(cascade_path):
                    self.cascade = cv2.CascadeClassifier(cascade_path)
                    self.use_haar = True
        except Exception:
            pass

    def detect(self, image: np.ndarray) -> Tuple[List, List]:
        bboxs, scores = [], []
        img_h, img_w = image.shape[:2]

        if self.use_dnn:
            try:
                blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0))
                self.net.setInput(blob)
                detections = self.net.forward()
                for i in range(detections.shape[2]):
                    confidence = float(detections[0, 0, i, 2])
                    if confidence > 0.4:
                        x1 = max(0.0, float(detections[0, 0, i, 3]))
                        y1 = max(0.0, float(detections[0, 0, i, 4]))
                        x2 = min(1.0, float(detections[0, 0, i, 5]))
                        y2 = min(1.0, float(detections[0, 0, i, 6]))
                        if x2 > x1 and y2 > y1:
                            bboxs.append([x1, y1, x2, y2])
                            scores.append(confidence)
                if len(bboxs) > 0:
                    return bboxs, scores
            except Exception:
                pass

        if self.use_haar:
            try:
                gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
                faces = self.cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
                for (x, y, w, h) in faces:
                    bboxs.append([float(x / img_w), float(y / img_h), float((x + w) / img_w), float((y + h) / img_h)])
                    scores.append(1.0)
                if len(bboxs) > 0:
                    return bboxs, scores
            except Exception:
                pass

        # 100% crash-proof fallback: Upper-center 1:1 face framing
        fx1 = 0.35
        fy1 = 0.15
        fx2 = 0.65
        fy2 = 0.55
        bboxs.append([fx1, fy1, fx2, fy2])
        scores.append(0.8)
        return bboxs, scores

    def __call__(self, image: np.ndarray) -> Tuple[List, List]:
        return self.detect(image)
''')
    except Exception:
        pass

# 5. facecrop.py (Strict 1:1 Square Crop Guarantee)
facecrop_py = "/content/SoulX-FlashHead/flash_head/utils/facecrop.py"
if os.path.exists(facecrop_py):
    try:
        with open(facecrop_py, "w") as _f:
            _f.write('''#!/usr/bin/env python3
import os
import json
from PIL import Image
import numpy as np

from flash_head.utils.cpu_face_handler import CPUFaceHandler

def get_scaled_bbox(
    bbox, img_w, img_h, ratio: float = 1.0, face_image: Image.Image = None
):
    """
    Computes a guaranteed strict 1:1 square crop centered on the face.
    Ensures width == height, shifts safely away from edges, and enforces even dimensions.
    """
    x1, y1, x2, y2 = bbox
    center_x = (x1 + x2) / 2.0
    center_y = (y1 + y2) / 2.0
    face_w = max(x2 - x1, 10.0)

    # Calculate target 1:1 square side length
    max_side = min(img_w, img_h)
    target_side = int(min(face_w * ratio, max_side))
    target_side = max(target_side, 64)

    # Slight downward bias so the face occupies the upper-middle of the square crop
    shift_y = target_side * 0.05
    center_y = center_y + shift_y

    half_side = target_side / 2.0
    new_x1 = int(round(center_x - half_side))
    new_y1 = int(round(center_y - half_side))

    # Shift bounding box away from edges so it doesn't get clipped into a non-square
    if new_x1 < 0:
        new_x1 = 0
    elif new_x1 + target_side > img_w:
        new_x1 = img_w - target_side

    if new_y1 < 0:
        new_y1 = 0
    elif new_y1 + target_side > img_h:
        new_y1 = img_h - target_side

    new_x2 = new_x1 + target_side
    new_y2 = new_y1 + target_side

    # Ensure even dimensions for video codecs
    if (new_x2 - new_x1) % 2 != 0:
        new_x2 -= 1
    if (new_y2 - new_y1) % 2 != 0:
        new_y2 -= 1

    scaled_bbox = [int(new_x1), int(new_y1), int(new_x2), int(new_y2)]

    try:
        with open("/tmp/_flashhead_crop_bbox.json", "w") as _bf:
            json.dump({"bbox": scaled_bbox, "img_w": img_w, "img_h": img_h}, _bf)
    except Exception:
        pass

    crop_face = face_image.crop(scaled_bbox)
    return crop_face

def process_image(
    input_path,
    face_ratio=1.8,
    target_size=(512, 512),
):
    """
    Detects face, crops a clean 1:1 square, and resizes to target_size.
    """
    face_detector = CPUFaceHandler()
    if not os.path.isfile(input_path):
        raise ValueError(f"File not found: {input_path}")

    try:
        image = Image.open(input_path).convert("RGB")
        image_rgb = np.array(image)
        img_h, img_w = image_rgb.shape[:2]

        boxes, scores = face_detector(image_rgb)
        if len(boxes) == 0:
            boxes = [[0.35, 0.15, 0.65, 0.55]]

        boxes_abs = [
            boxes[0][0] * img_w,
            boxes[0][1] * img_h,
            boxes[0][2] * img_w,
            boxes[0][3] * img_h
        ]

        crop_face = get_scaled_bbox(boxes_abs, img_w, img_h, face_ratio, image)
        crop_face = crop_face.resize(target_size, Image.Resampling.LANCZOS)
        return crop_face
    except Exception as e:
        raise ValueError(f"Error processing {input_path}: {e}")
''')
    except Exception:
        pass

import torch
import librosa
import imageio
import gradio as gr

# Configure Loguru to be clean and informative
logger.remove()
logger.add(sys.stdout, format="<green>{time:HH:mm:ss}</green> | <level>{level: <7}</level> | {message}", level="INFO")

# Import FlashHead modules
from flash_head.inference import get_pipeline, get_base_data, get_infer_params, get_audio_embedding, run_pipeline

# Global pipeline cache
pipeline = None
loaded_ckpt_dir = None
loaded_wav2vec_dir = None
loaded_model_type = None

# Hardware-accelerated FFmpeg encoder detector
def get_best_ffmpeg_encoder():
    """Detects if h264_nvenc hardware acceleration is available."""
    try:
        res = subprocess.run(["ffmpeg", "-encoders"], capture_output=True, text=True, timeout=3)
        if "h264_nvenc" in res.stdout:
            return ["-c:v", "h264_nvenc", "-preset", "p1", "-cq", "19", "-pix_fmt", "yuv420p"]
    except Exception:
        pass
    return ["-c:v", "libx264", "-preset", "ultrafast", "-tune", "fastdecode", "-crf", "19", "-pix_fmt", "yuv420p", "-threads", "0"]

# Ultra-Fast Video Compositor & Exporter
def save_video_to_file(frames_list, video_path, audio_path, fps, original_image_path=None, output_format="Preserve Original Aspect Ratio (16:9 / 9:16 Composite)"):
    """
    Ultra-fast video rendering and aspect-ratio preservation.
    Supports direct 1:1 square export and hardware-accelerated 16:9 / 9:16 composite.
    """
    os.makedirs(os.path.dirname(video_path), exist_ok=True)
    temp_face_video = video_path.replace(".mp4", "_face512.mp4")

    t_render_start = time.time()
    # 1. Write the 512x512 face video directly (fast, no heavy memory churn)
    with imageio.get_writer(
        temp_face_video, format="mp4", mode="I", fps=fps, codec="h264",
        ffmpeg_params=["-preset", "ultrafast", "-crf", "18", "-bf", "0", "-threads", "0"]
    ) as writer:
        for frames in frames_list:
            frames_np = frames.numpy().astype(np.uint8)
            for i in range(frames_np.shape[0]):
                writer.append_data(frames_np[i])

    # 2. Check if user requested 1:1 Square Output directly
    if output_format == "1:1 Square (Cropped Face Only)":
        cmd = [
            "ffmpeg", "-y", "-i", temp_face_video, "-i", audio_path,
            "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
            "-shortest", video_path
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        if os.path.exists(temp_face_video):
            os.remove(temp_face_video)
        print(f"✅ 1:1 Square face video exported in {time.time() - t_render_start:.1f}s!", flush=True)
        return video_path

    # 3. Check Aspect Ratio and Compositing
    bbox_file = "/tmp/_flashhead_crop_bbox.json"
    should_composite = False
    crop_info = None

    if original_image_path and os.path.exists(original_image_path):
        try:
            with Image.open(original_image_path) as img:
                orig_w, orig_h = img.size
            aspect = max(orig_w, orig_h) / max(min(orig_w, orig_h), 1)

            if aspect > 1.05:  # Non-square: 16:9, 9:16, etc.
                should_composite = True
                if os.path.exists(bbox_file):
                    try:
                        with open(bbox_file, "r") as f:
                            saved_data = json.load(f)
                        crop_info = {
                            "bbox": saved_data["bbox"],
                            "orig_w": orig_w,
                            "orig_h": orig_h
                        }
                    except Exception:
                        crop_info = None

                if crop_info is None:
                    side = min(orig_w, orig_h)
                    if orig_w > orig_h:  # 16:9 widescreen
                        cx1 = (orig_w - side) // 2
                        cy1 = 0
                    else:  # 9:16 portrait
                        cx1 = 0
                        cy1 = (orig_h - side) // 2
                    cx2 = cx1 + side
                    cy2 = cy1 + side
                    crop_info = {
                        "bbox": [cx1, cy1, cx2, cy2],
                        "orig_w": orig_w,
                        "orig_h": orig_h
                    }
        except Exception:
            should_composite = False

    # Clean up temporary bbox file
    if os.path.exists(bbox_file):
        try:
            os.remove(bbox_file)
        except Exception:
            pass

    # 4. Hardware-Accelerated Native FFmpeg Compositing
    if should_composite and crop_info:
        crop_x1, crop_y1, crop_x2, crop_y2 = crop_info["bbox"]
        crop_w = max(crop_x2 - crop_x1, 2)
        crop_h = max(crop_y2 - crop_y1, 2)
        # Ensure even dimensions
        crop_w = crop_w & ~1
        crop_h = crop_h & ~1

        ar_label = "16:9 (Widescreen)" if orig_w > orig_h else "9:16 (Vertical Portrait)"
        print(f"\n📐 Aspect Ratio System: Detected {ar_label} · Resolution: {orig_w}x{orig_h}", flush=True)
        print(f"   Executing hardware-accelerated composite into ({crop_x1},{crop_y1})-({crop_x2},{crop_y2}) [{crop_w}x{crop_h}]...", flush=True)

        encoder_args = get_best_ffmpeg_encoder()
        cmd = [
            "ffmpeg", "-y",
            "-loop", "1", "-i", original_image_path,
            "-i", temp_face_video,
            "-i", audio_path,
            "-filter_complex",
            f"[1:v]scale={crop_w}:{crop_h}[face]; [0:v][face]overlay={crop_x1}:{crop_y1}:shortest=1[outv]",
            "-map", "[outv]", "-map", "2:a",
        ] + encoder_args + [
            "-c:a", "aac", "-b:a", "192k",
            "-shortest", video_path
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        print(f"✅ Video preserved at native {orig_w}x{orig_h} in {time.time() - t_render_start:.1f}s!", flush=True)
    else:
        # Standard square output muxing
        cmd = [
            "ffmpeg", "-y", "-i", temp_face_video, "-i", audio_path,
            "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
            "-shortest", video_path
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

    if os.path.exists(temp_face_video):
        os.remove(temp_face_video)

    return video_path

# Main High-Speed Single-GPU Inference Pipeline
def run_inference(
    ckpt_dir,
    wav2vec_dir,
    model_type,
    cond_image,
    audio_path,
    audio_encode_mode,
    seed,
    use_face_crop,
    sampling_steps=4,
    output_format="Preserve Original Aspect Ratio (16:9 / 9:16 Composite)",
    progress=gr.Progress(track_tqdm=True),
):
    global pipeline, loaded_ckpt_dir, loaded_wav2vec_dir, loaded_model_type

    if not cond_image or not os.path.exists(cond_image):
        raise gr.Error("Please upload a valid portrait image.")
    if not audio_path or not os.path.exists(audio_path):
        raise gr.Error("Please upload a valid speech audio file.")

    start_total_time = time.time()
    print(f"\n🎬 [SoulX FlashHead] Starting video generation...", flush=True)
    print(f"📷 Image: {cond_image} | 🎵 Audio: {audio_path}", flush=True)

    # 1. Pipeline Caching & Loading
    if (pipeline is None or loaded_ckpt_dir != ckpt_dir or 
        loaded_wav2vec_dir != wav2vec_dir or loaded_model_type != model_type):
        progress(0.05, desc="⚡ Loading SoulX FlashHead Pipeline into VRAM...")
        print(f"⏳ Loading pipeline: {model_type} (FP16 Engine)...", flush=True)
        try:
            pipeline = get_pipeline(world_size=1, ckpt_dir=ckpt_dir, model_type=model_type, wav2vec_dir=wav2vec_dir)
            loaded_ckpt_dir = ckpt_dir
            loaded_wav2vec_dir = wav2vec_dir
            loaded_model_type = model_type
            print("✅ Pipeline loaded into GPU VRAM.", flush=True)
        except Exception as e:
            logger.error(f"Failed to load pipeline: {e}")
            raise gr.Error(f"Model loading failed: {e}")

    # Set inference parameters and steps
    infer_params = get_infer_params()
    infer_params["sample_steps"] = int(sampling_steps)

    # 2. Prepare Data & Face Detection
    progress(0.12, desc="👤 Preparing Reference Face & Latents...")
    base_seed = int(seed) if seed >= 0 else int(np.random.randint(0, 2147483647))
    print(f"🎲 Seed: {base_seed} ({'Random' if seed < 0 else 'Manual'})", flush=True)
    try:
        pipeline.prepare_params(
            cond_image_path_or_dir=cond_image,
            target_size=(infer_params["height"], infer_params["width"]),
            frame_num=infer_params["frame_num"],
            motion_frames_num=infer_params["motion_frames_num"],
            sampling_steps=int(sampling_steps),
            seed=base_seed,
            shift=infer_params["sample_shift"],
            color_correction_strength=infer_params["color_correction_strength"],
            use_face_crop=use_face_crop,
        )
    except Exception as e:
        logger.error(f"Error preparing base data: {e}")
        raise gr.Error(f"Input image processing failed: {e}")

    sample_rate = infer_params["sample_rate"]
    tgt_fps = infer_params["tgt_fps"]
    cached_audio_duration = infer_params["cached_audio_duration"]
    frame_num = infer_params["frame_num"]
    motion_frames_num = infer_params["motion_frames_num"]
    slice_len = frame_num - motion_frames_num

    # 3. Audio Feature Extraction
    progress(0.18, desc="🎵 Extracting Wav2Vec2 Audio Embeddings...")
    try:
        human_speech_array_all, _ = librosa.load(audio_path, sr=sample_rate, mono=True)
    except Exception as e:
        raise gr.Error(f"Audio loading failed: {e}")

    human_speech_array_slice_len = slice_len * sample_rate // tgt_fps
    human_speech_array_frame_num = frame_num * sample_rate // tgt_fps
    generated_list = []

    # 4. High-Speed Generation Loop with Live Console Progress Bar
    if audio_encode_mode == "once":
        remainder = (len(human_speech_array_all) - human_speech_array_frame_num) % human_speech_array_slice_len
        if remainder > 0:
            pad_length = human_speech_array_slice_len - remainder
            human_speech_array_all = np.concatenate([
                human_speech_array_all, np.zeros(pad_length, dtype=human_speech_array_all.dtype)
            ])

        audio_embedding_all = get_audio_embedding(pipeline, human_speech_array_all)
        audio_embedding_chunks_list = [
            audio_embedding_all[:, i * slice_len: i * slice_len + frame_num].contiguous()
            for i in range((audio_embedding_all.shape[1] - frame_num) // slice_len)
        ]
        total_chunks = len(audio_embedding_chunks_list)
        print(f"⚡ Generating talking head video ({total_chunks} chunks · {model_type.upper()} model · {sampling_steps} steps)...", flush=True)

        pbar = tqdm(total=total_chunks, desc=f"⚡ Generating ({model_type.upper()})", unit="chunk", file=sys.stdout, leave=True, dynamic_ncols=True)
        for chunk_idx, audio_embedding_chunk in enumerate(audio_embedding_chunks_list):
            progress_pct = 0.22 + 0.68 * (chunk_idx / total_chunks)
            progress(progress_pct, desc=f"🎬 Generating: Chunk {chunk_idx + 1}/{total_chunks} ({int(progress_pct * 100)}%)")

            video = run_pipeline(pipeline, audio_embedding_chunk)
            if chunk_idx != 0:
                video = video[motion_frames_num:]

            generated_list.append(video.cpu())
            pbar.update(1)
            sys.stdout.flush()
        pbar.close()

    else:
        cached_audio_length_sum = sample_rate * cached_audio_duration
        audio_end_idx = cached_audio_duration * tgt_fps
        audio_start_idx = audio_end_idx - frame_num
        audio_dq = deque([0.0] * cached_audio_length_sum, maxlen=cached_audio_length_sum)

        remainder = len(human_speech_array_all) % human_speech_array_slice_len
        if remainder > 0:
            pad_length = human_speech_array_slice_len - remainder
            human_speech_array_all = np.concatenate([
                human_speech_array_all, np.zeros(pad_length, dtype=human_speech_array_all.dtype)
            ])

        human_speech_array_slices = human_speech_array_all.reshape(-1, human_speech_array_slice_len)
        total_chunks = len(human_speech_array_slices)
        print(f"⚡ Streaming talking head video ({total_chunks} chunks · {model_type.upper()} model)...", flush=True)

        pbar = tqdm(total=total_chunks, desc=f"⚡ Streaming ({model_type.upper()})", unit="chunk", file=sys.stdout, leave=True, dynamic_ncols=True)
        for chunk_idx, human_speech_array in enumerate(human_speech_array_slices):
            progress_pct = 0.22 + 0.68 * (chunk_idx / total_chunks)
            progress(progress_pct, desc=f"🎬 Generating: Chunk {chunk_idx + 1}/{total_chunks} ({int(progress_pct * 100)}%)")

            audio_dq.extend(human_speech_array.tolist())
            audio_array = np.array(audio_dq)
            audio_embedding = get_audio_embedding(pipeline, audio_array, audio_start_idx, audio_end_idx)

            video = run_pipeline(pipeline, audio_embedding)
            video = video[motion_frames_num:]
            generated_list.append(video.cpu())
            pbar.update(1)
            sys.stdout.flush()
        pbar.close()

    # 5. Fast Video Rendering & Aspect Ratio Compositing
    progress(0.93, desc="📦 Hardware Video Rendering & Audio Multiplexing...")
    output_dir = "gradio_results"
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = os.path.join(output_dir, f"soulx_head_{timestamp}.mp4")

    final_video_path = save_video_to_file(
        generated_list, save_path, audio_path, fps=tgt_fps,
        original_image_path=cond_image, output_format=output_format
    )

    elapsed = time.time() - start_total_time
    speed_per_chunk = elapsed / max(total_chunks, 1)
    status_msg = f"✅ Video generated in {elapsed:.1f}s ({total_chunks} chunks · {speed_per_chunk:.2f}s/chunk)!"
    print(f"\n{status_msg}", flush=True)
    print(f"📁 Output file: {final_video_path}\n", flush=True)
    progress(1.0, desc="✅ Complete!")

    return final_video_path, status_msg

# Multi-GPU Runner
def run_multi_gpu_inference(
    gpu_ids, ckpt_dir, wav2vec_dir, model_type, cond_image, audio_path,
    audio_encode_mode, use_face_crop, seed, progress=gr.Progress()
):
    gpu_list = [x.strip() for x in gpu_ids.split(",") if x.strip()]
    num_gpus = len(gpu_list)
    if num_gpus == 0:
        raise gr.Error("Please specify at least one GPU ID (e.g. '0' or '0,1').")

    output_dir = "gradio_results_multigpu"
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = os.path.abspath(os.path.join(output_dir, f"soulx_head_{timestamp}.mp4"))

    cmd = [
        "torchrun", f"--nproc_per_node={num_gpus}", "generate_video.py",
        "--ckpt_dir", ckpt_dir,
        "--wav2vec_dir", wav2vec_dir,
        "--model_type", model_type,
        "--cond_image", cond_image,
        "--audio_path", audio_path,
        "--audio_encode_mode", audio_encode_mode,
        "--use_face_crop", str(use_face_crop),
        "--base_seed", str(int(seed)),
        "--save_file", save_path,
    ]

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = ",".join(gpu_list)
    progress(0.1, desc="⚡ Launching Multi-GPU torchrun...")
    proc = subprocess.run(cmd, env=env)
    if proc.returncode != 0:
        raise gr.Error(f"Multi-GPU inference failed with return code {proc.returncode}")

    # Aspect ratio check for multi-gpu output
    save_video_to_file([], save_path, audio_path, fps=25, original_image_path=cond_image)
    return save_path, f"✅ Multi-GPU video saved to {save_path}"

# ═══════════════════════════════════════════════════════════════════════════
# Gradio Studio UI (Strict AIQUEST Academy Branding)
# ═══════════════════════════════════════════════════════════════════════════

CUSTOM_CSS = """
.brand-header {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    border-radius: 16px;
    padding: 24px;
    margin-bottom: 20px;
    box-shadow: 0 8px 24px rgba(102, 126, 234, 0.25);
    text-align: center;
    color: white;
}
.brand-title {
    font-size: 28px !important;
    font-weight: 800 !important;
    margin: 0 0 6px 0 !important;
    color: white !important;
    letter-spacing: -0.5px;
}
.brand-subtitle {
    font-size: 14px;
    opacity: 0.95;
    margin-bottom: 16px;
}
.social-buttons {
    display: flex;
    justify-content: center;
    gap: 12px;
    flex-wrap: wrap;
}
.social-btn {
    display: inline-flex;
    align-items: center;
    padding: 8px 16px;
    border-radius: 8px;
    font-weight: 600;
    font-size: 13px;
    text-decoration: none !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease;
}
.social-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 4px 12px rgba(0, 0, 0, 0.2);
}
.youtube-btn {
    background: #ff0000;
    color: white !important;
}
.x-btn {
    background: #000000;
    color: white !important;
}
#gen-btn {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
}
#stop-btn {
    background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
}
#clear-btn {
    background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
}
.footer-text {
    text-align: center;
    padding: 20px;
    margin-top: 25px;
    border-top: 1px solid #e5e7eb;
    color: #6b7280;
    font-size: 13px;
}
"""

with gr.Blocks(title="SoulX FlashHead Studio - AIQUEST Academy", css=CUSTOM_CSS) as app:
    # AIQUEST Branding Header
    gr.HTML(
        """
        <div class="brand-header">
            <h1 class="brand-title">🎬 SoulX FlashHead Studio</h1>
            <div class="brand-subtitle">Real-Time Audio-Driven Portrait Animation · Google Colab T4 FP16 Edition</div>
            <div class="social-buttons">
                <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn youtube-btn">
                    ▶ Subscribe on YouTube
                </a>
                <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">
                    Follow on X
                </a>
            </div>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group():
                gr.Markdown("### 📷 Portrait & Speech Inputs")
                cond_image_input = gr.Image(
                    label="Condition Portrait Image (16:9, 9:16, or Square)",
                    type="filepath",
                    value="examples/girl.png" if os.path.exists("examples/girl.png") else None,
                    height=280
                )
                audio_path_input = gr.Audio(
                    label="Input Speech Audio",
                    type="filepath",
                    value="examples/podcast_sichuan_16k.wav" if os.path.exists("examples/podcast_sichuan_16k.wav") else None
                )

            with gr.Row():
                gen_btn = gr.Button("🎬 Generate Video", variant="primary", size="lg", elem_id="gen-btn")
                stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

            with gr.Accordion("⚙️ Advanced Settings & Performance Options", open=False):
                model_type_input = gr.Dropdown(
                    label="Model Variant",
                    choices=[
                        ("Lite (Fastest · 1.9s/chunk on T4)", "lite"),
                        ("Pro (High Fidelity · Wan 2.1 DiT)", "pro")
                    ],
                    value="lite",
                    info="Lite variant uses LTX-Video VAE for real-time inference. Pro provides higher fidelity."
                )
                sampling_steps_input = gr.Slider(
                    label="Sampling Steps",
                    minimum=4,
                    maximum=8,
                    step=2,
                    value=4,
                    info="Default: 4 steps (preserves full model convergence and highest fidelity for Lite & Pro)."
                )
                output_format_input = gr.Radio(
                    label="📐 Output Video Format",
                    choices=[
                        "Preserve Original Aspect Ratio (16:9 / 9:16 Composite)",
                        "1:1 Square (Cropped Face Only)"
                    ],
                    value="Preserve Original Aspect Ratio (16:9 / 9:16 Composite)",
                    info="Choose whether to composite the animated face back into the 16:9/9:16 background or export the 1:1 face directly in sub-second time."
                )
                audio_encode_mode_input = gr.Radio(
                    label="Audio Encode Mode",
                    choices=[("Once (Fastest Batch)", "once"), ("Stream (Chunk-by-Chunk)", "stream")],
                    value="once",
                    info="'Once' encodes audio in a single pass for peak generation speed."
                )
                use_face_crop_input = gr.Checkbox(
                    label="Enable Smart Face Crop & Centering",
                    value=True,
                    info="Detects face, crops a clean 1:1 square centered on the face for optimal animation quality."
                )
                seed_input = gr.Number(
                    label="Random Seed (-1 for random)",
                    value=-1,
                    precision=0,
                    info="Default: -1 picks a unique random seed on every generation."
                )
                mode_input = gr.Radio(
                    choices=["Single GPU", "Multi-GPU"],
                    value="Single GPU",
                    label="Execution Mode",
                    visible=False
                )
                gpu_ids_input = gr.Textbox(
                    label="GPU IDs",
                    value="0",
                    visible=False
                )
                ckpt_dir_input = gr.Textbox(
                    label="Model Checkpoint Directory",
                    value="/content/models/SoulX-FlashHead-1_3B",
                    visible=False
                )
                wav2vec_dir_input = gr.Textbox(
                    label="Wav2Vec Directory",
                    value="/content/models/wav2vec2-base-960h",
                    visible=False
                )

        with gr.Column(scale=1):
            gr.Markdown("### 📺 Generated Talking Head Video")
            video_output = gr.Video(label="Rendered Video Result", height=420)
            status_output = gr.Markdown("⏳ Ready to generate. Click **Generate Video** to begin.")

    # AIQUEST Branding Footer
    gr.HTML(
        """
        <div class="footer-text">
            ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · <a href="https://aiquest.site" target="_blank" style="color:#667eea; text-decoration:none;">aiquest.site</a> · © All rights reserved
        </div>
        """
    )

    # Event Dispatcher
    def dispatch_inference(mode, gpu_ids, ckpt, wav2vec, model_type, img, audio, enc_mode, seed, use_face_crop, steps, out_fmt):
        if mode == "Single GPU":
            return run_inference(
                ckpt, wav2vec, model_type, img, audio, enc_mode, seed,
                use_face_crop, sampling_steps=steps, output_format=out_fmt
            )
        else:
            return run_multi_gpu_inference(gpu_ids, ckpt, wav2vec, model_type, img, audio, enc_mode, use_face_crop, seed)

    # Event Bindings
    gen_event = gen_btn.click(
        fn=dispatch_inference,
        inputs=[
            mode_input, gpu_ids_input, ckpt_dir_input, wav2vec_dir_input,
            model_type_input, cond_image_input, audio_path_input,
            audio_encode_mode_input, seed_input, use_face_crop_input,
            sampling_steps_input, output_format_input
        ],
        outputs=[video_output, status_output]
    )

    stop_btn.click(fn=None, cancels=[gen_event])

    clear_btn.click(
        fn=lambda: (None, "🗑️ Cleared inputs and video output."),
        outputs=[video_output, status_output]
    )

# Launch Gradio Studio with Dual Public Access: Cloudflare Tunnel & Gradio Share
def start_cloudflare_tunnel(port=7860):
    """
    Start Cloudflare tunnel in background and extract public trycloudflare.com URL.
    Returns: (subprocess.Popen or None, tunnel_url or None)
    """
    cf_bin = "/usr/local/bin/cloudflared"
    if not os.path.exists(cf_bin):
        import shutil
        cf_bin = shutil.which("cloudflared")

    if not cf_bin or not os.path.exists(cf_bin):
        try:
            subprocess.run(
                "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared > /dev/null 2>&1",
                shell=True,
                check=False
            )
            cf_bin = "/usr/local/bin/cloudflared"
        except Exception:
            return None, None

    if not cf_bin or not os.path.exists(cf_bin):
        return None, None

    log_file = "/tmp/cloudflared.log"
    try:
        if os.path.exists(log_file):
            os.remove(log_file)
        log_f = open(log_file, "w")
        proc = subprocess.Popen(
            [cf_bin, "tunnel", "--url", f"http://127.0.0.1:{port}"],
            stdout=log_f,
            stderr=subprocess.STDOUT
        )
    except Exception:
        return None, None

    tunnel_url = None
    for _ in range(30):
        time.sleep(0.5)
        if os.path.exists(log_file):
            try:
                with open(log_file, "r") as f:
                    content = f.read()
                    matches = re.findall(r"https://[a-zA-Z0-9.-]+[.]trycloudflare[.]com", content)
                    if matches:
                        tunnel_url = matches[0]
                        break
            except Exception:
                pass

    return proc, tunnel_url

SERVER_PORT = 7860

print("=" * 65)
print("🚀 Initializing SoulX FlashHead Studio & Cloudflare Tunnel...")
print("=" * 65)

# 1. Start Cloudflare Tunnel in background
cf_proc, cf_url = start_cloudflare_tunnel(port=SERVER_PORT)

# 2. Launch Gradio Studio
app.queue(max_size=20, default_concurrency_limit=1)
launch_res = app.launch(
    share=True,
    inline=False,
    debug=False,
    show_error=True,
    server_port=SERVER_PORT,
    prevent_thread_lock=True
)

share_url = getattr(app, "share_url", None)
if not share_url and isinstance(launch_res, tuple) and len(launch_res) >= 3:
    share_url = launch_res[2]

# If Cloudflare tunnel needed extra seconds, poll once more
if not cf_url and os.path.exists("/tmp/cloudflared.log"):
    try:
        with open("/tmp/cloudflared.log", "r") as f:
            m = re.findall(r"https://[a-zA-Z0-9.-]+[.]trycloudflare[.]com", f.read())
            if m:
                cf_url = m[0]
    except Exception:
        pass

print("\n" + "=" * 65)
print("🎬 SoulX FlashHead Studio is LIVE!")
print("=" * 65)
if cf_url:
    print(f"🌐 Cloudflare Tunnel (Recommended): {cf_url}")
else:
    print("🌐 Cloudflare Tunnel:                (Connecting... check /tmp/cloudflared.log)")
if share_url:
    print(f"🔗 Gradio Public Share (Backup):    {share_url}")
print(f"🖥️ Local Instance URL:               http://127.0.0.1:{SERVER_PORT}")
print("⚡ Streaming real-time generation progress and logs below...")
print("=" * 65 + "\n")

# Keep the cell active and spinning so real-time logs and progress bars stream to notebook
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Stopped SoulX FlashHead Studio.")
    if cf_proc:
        cf_proc.terminate()

---

<div align="center">
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logo=white" />
  </a>
</div>
<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---